## P31 — book figure
Streamlined from `p31.ipynb`. Two outputs for one episode:
1. Camera grid — rows are time, columns are camera views
2. Control-signal time series, with dots at the frames shown in the grid

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from tqdm import tqdm

## Config

In [ ]:
EPISODE_IDX = 0
FRAME_STEP  = 30      # show every Nth frame
FRAMES      = None    # or an explicit list, e.g. [0, 45, 90, 200] (overrides FRAME_STEP)

# Cameras (grid columns)
ONLY_MODEL_CAMS = False   # True -> just the 3 views pi0 actually sees (drops cam_low)
MODEL_CAMS = ["cam_high", "cam_left_wrist", "cam_right_wrist"]
ALL_CAMS   = ["cam_high", "cam_low", "cam_left_wrist", "cam_right_wrist"]
CAM_TITLES = {"cam_high": "High", "cam_low": "Low",
              "cam_left_wrist": "Left wrist", "cam_right_wrist": "Right wrist"}

# Grid look
GRID_LABELS = True    # column titles + frame index on each row
COL_WIDTH   = 2.5     # inches per image column
GRID_GAP    = 0.03    # gap between images, as a fraction of image size
GRID_DPI    = 300     # 2.5in * 300dpi = 750px per image (source frames are 640x480)

# Control signal
SIGNAL        = "pred"   # "pred" = pi0 output, "gt" = actions recorded in the dataset
REPLAN_EVERY  = 5        # pred only: run pi0 every N frames, keep the first N actions of each chunk
JOINTS        = list(range(14))   # which of the 14 action dims to plot
SHOW_LEGEND   = True
TS_FIGSIZE    = (16, 6)
LINEWIDTH     = 3
DOT_SIZE      = 9
YLIM          = (-1.0, 1.3)
FONTSIZE      = 18
TS_DPI        = 300

PROMPT = "uncap the pen"
checkpoint_dir = "/home/stephen/robots/openpi/pi0_aloha_pen_uncap_pytorch"
save_dir = "/home/stephen/robots/p31/book_figure"
os.makedirs(save_dir, exist_ok=True)

CHILL_BROWN = '#948979'
labels_all = ["L Waist", "L Shoulder", "L Elbow", "L Forearm", "L Wrist angle", "L Wrist rotate", "L Gripper",
              "R Waist", "R Shoulder", "R Elbow", "R Forearm", "R Wrist angle", "R Wrist rotate", "R Gripper"]

In [ ]:
def style_ax(ax, color=CHILL_BROWN):
    ax.spines[:].set_color(color)
    ax.tick_params(colors=color, which='both')
    ax.xaxis.label.set_color(color)
    ax.yaxis.label.set_color(color)
    ax.title.set_color(color)

def style_cbar(cbar, color=CHILL_BROWN):
    cbar.outline.set_edgecolor(color)
    cbar.ax.tick_params(colors=color, which='both')
    cbar.ax.yaxis.label.set_color(color)
    cbar.ax.xaxis.label.set_color(color)
    cbar.ax.title.set_color(color)

## Load episode

In [ ]:
ds = load_dataset("physical-intelligence/aloha_pen_uncap_diverse", split="train")

row_indices = np.where(np.array(ds["episode_index"]) == EPISODE_IDX)[0]
episode = ds.select(row_indices)
n_frames = episode.num_rows

frame_idxs = list(FRAMES) if FRAMES is not None else list(range(0, n_frames, FRAME_STEP))
cams = MODEL_CAMS if ONLY_MODEL_CAMS else ALL_CAMS
tag = f"ep{EPISODE_IDX:03d}"
print(f"{n_frames} frames, showing {len(frame_idxs)}: {frame_idxs}")

## 1. Camera grid

In [ ]:
n_rows, n_cols = len(frame_idxs), len(cams)
w, h = episode[0][f"observation.images.{cams[0]}"].size   # (640, 480)

fig, axes = plt.subplots(n_rows, n_cols, squeeze=False,
                         figsize=(n_cols * COL_WIDTH, n_rows * COL_WIDTH * h / w),
                         gridspec_kw=dict(wspace=GRID_GAP, hspace=GRID_GAP * w / h))

for r, fi in enumerate(frame_idxs):
    frame = episode[fi]
    for c, cam in enumerate(cams):
        ax = axes[r, c]
        ax.imshow(frame[f"observation.images.{cam}"])
        ax.axis("off")
        if GRID_LABELS and r == 0:
            ax.set_title(CAM_TITLES[cam], color=CHILL_BROWN, fontsize=11)
        if GRID_LABELS and c == 0:
            ax.text(-0.04, 0.5, str(fi), transform=ax.transAxes, rotation=90,
                    ha="right", va="center", color=CHILL_BROWN, fontsize=11)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
fig.savefig(f"{save_dir}/{tag}_camera_grid.png", dpi=GRID_DPI, transparent=True,
            bbox_inches="tight", pad_inches=0.02)
# SVG works too (handy if you want each frame as a separate object in Illustrator),
# but the frames are just embedded as PNGs inside it — no sharper, much bigger file.
# fig.savefig(f"{save_dir}/{tag}_camera_grid.svg", dpi=GRID_DPI, transparent=True,
#             bbox_inches="tight", pad_inches=0.02)
plt.show()

## 2. Control signal
`SIGNAL = "pred"` runs pi0 every `REPLAN_EVERY` frames and stitches together the first `REPLAN_EVERY` actions of each chunk (the "run and replace" scheme from p31). The result is cached, so restyling the plot doesn't re-run the model. `SIGNAL = "gt"` skips the model entirely.

In [ ]:
def frame_to_obs(frame):
    to_chw = lambda img: np.array(img).transpose(2, 0, 1)   # PIL HWC -> CHW
    return {
        "images": {cam: to_chw(frame[f"observation.images.{cam}"]) for cam in MODEL_CAMS},
        "state": np.array(frame["observation.state"]),
        "prompt": PROMPT,
    }

if SIGNAL == "gt":
    signal = np.array(episode["action"])
else:
    cache = f"{save_dir}/{tag}_pred_replan{REPLAN_EVERY}.npy"
    if os.path.exists(cache):
        signal = np.load(cache)
    else:
        from openpi.training import config as _config
        from openpi.policies import policy_config as _policy_config
        policy = _policy_config.create_trained_policy(_config.get_config("pi0_aloha_pen_uncap"), checkpoint_dir)

        fixed_noise = np.random.RandomState(42).randn(50, 32).astype(np.float32)
        chunks = []
        for i in tqdm(range(0, n_frames, REPLAN_EVERY)):
            actions = policy.infer(frame_to_obs(episode[i]), noise=fixed_noise)["actions"]
            chunks.append(actions[:REPLAN_EVERY])
        signal = np.concatenate(chunks)[:n_frames]
        np.save(cache, signal)

print(signal.shape)

In [ ]:
fig, ax = plt.subplots(figsize=TS_FIGSIZE)

for j in JOINTS:
    ax.plot(signal[:, j], linewidth=LINEWIDTH, label=labels_all[j],
            marker="o", markersize=DOT_SIZE, markevery=[f for f in frame_idxs if f < len(signal)])

ax.set_xlim(0, len(signal) - 1)
ax.set_ylim(*YLIM)
ax.set_xticks(frame_idxs)          # ticks line up with the grid rows
ax.set_xlabel("Frame", fontsize=FONTSIZE)
ax.tick_params(labelsize=FONTSIZE)
style_ax(ax)

if SHOW_LEGEND:
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=FONTSIZE * 0.75,
              frameon=False, labelcolor=CHILL_BROWN)

for ext in ("png", "svg"):
    fig.savefig(f"{save_dir}/{tag}_control_{SIGNAL}.{ext}", dpi=TS_DPI, transparent=True, bbox_inches="tight")
plt.show()